# Análisis comercial inmobiliario

*Proyecto guiado del programa de Análisis de Datos de TripleTen (Sprint 11), desarrollado en Power BI. El caso y los datasets vienen del curso; el modelo de datos, las medidas, las visualizaciones y las conclusiones son míos. Este notebook documenta el proceso y el archivo del dashboard (.pbix) se enlaza al final.*

El sector inmobiliario necesita evaluar su desempeño comercial para entender su crecimiento, su rentabilidad y el comportamiento de sus clientes. El proyecto consiste en construir un dashboard ejecutivo que permita analizar ventas, clientes y propiedades para apoyar decisiones basadas en datos.

## Datasets

- `hecho_ventas_propiedades`: tabla de hechos con las transacciones de venta (precio, cliente, propiedad, canal y fecha).
- `dim_clientes`: información y segmentación de los clientes.
- `dim_propiedades`: características de las propiedades (tipo, tamaño, ubicación, etc.).
- `dim_fecha`: tabla calendario que creo como parte del modelado.

## Preguntas que debe responder el dashboard

**Desempeño general**
- ¿Cuál es el ingreso total que genera la empresa?
- ¿Cuántas propiedades se han vendido?
- ¿Cuál es el precio promedio de venta?
- ¿Cuál es la comisión total generada?

**Análisis comercial**
- ¿Qué tipo de propiedad genera más ingresos?
- ¿Qué segmento de clientes compra más propiedades?
- ¿Qué canal de venta genera más ingresos?

**Análisis temporal**
- ¿Cómo evolucionan las ventas en el tiempo?
- ¿El negocio crece año contra año?
- ¿Cómo va el desempeño acumulado del año actual (YTD)?

**Cohortes de clientes**
- ¿Los clientes vuelven a comprar después de su primera compra?
- ¿Qué cohortes generan más compras o ingresos con el tiempo?

## Paso 1: Limpieza de datos

Antes de modelar validé y depuré la información.

**Tipos de datos:** `fecha_venta` en formato Date, los campos numéricos con el formato correcto y `porcentaje_comision` como porcentaje.

**Duplicados:** en `dim_clientes` y `dim_propiedades` no hay duplicados en las claves primarias, así que las mantuve como llaves primarias. Tampoco hay llaves huérfanas entre la tabla de hechos y las dimensiones.

## Paso 2: Tabla calendario

Creé la tabla `dim_fecha` en Power BI con las funciones `CALENDAR` y `ADDCOLUMNS`. El rango de fechas se construye de forma dinámica, desde la fecha mínima hasta la máxima de ventas en `hecho_ventas_propiedades`, e incluye año, nombre del mes, número de mes y año-mes (`YYYY-MM`).

## Paso 3: Modelado de datos

Armé un modelo en estrella con `hecho_ventas_propiedades` como tabla central y `dim_clientes`, `dim_propiedades` y `dim_fecha` como dimensiones. Todas las relaciones son de uno a muchos, con dirección de filtro simple y están activas.

## Paso 4: Medidas

**Medidas base:** Ingreso Total (suma de `precio_venta`), Cantidad de Ventas (con `DISTINCTCOUNT`), Ticket Promedio y Comisión Total.

**Medidas con contexto de filtro:** participación del ingreso por tipo de propiedad, por canal de venta y por segmento de cliente. Usan `CALCULATE` con `ALL` para quitar el filtro de la dimensión correspondiente en el denominador, mientras el numerador lo conserva. Las uso como tooltip en los gráficos.

**Inteligencia de tiempo** (sobre `dim_fecha`): ventas acumuladas del año (`TOTALYTD`) y del mes (`TOTALMTD`), ventas del año anterior (`SAMEPERIODLASTYEAR`) y crecimiento año contra año (YoY).

**Cohortes:** columnas calculadas en `hecho_ventas_propiedades` con la primera compra por cliente (fecha mínima, usando `ALLEXCEPT`), el mes de cohorte, el mes de venta y los meses transcurridos desde la cohorte. Con ellas armé la matriz de cohortes. Además agregué una medida de retención.

## Paso 5: Estructura del reporte

El dashboard tiene tres páginas.

**1. Overview ejecutivo.** Cuatro KPIs (ingreso total, cantidad de ventas, ticket promedio y comisión total), la tendencia de ventas en el tiempo, ingresos o ventas por ciudad y el crecimiento año contra año.

**2. Análisis comercial.** Ingreso por tipo de propiedad, ventas o ingresos por canal y por segmento de cliente, y una tabla con formato condicional tipo semáforo (tipo de propiedad, ingreso total, cantidad de ventas y ticket promedio) para resaltar las categorías de mejor desempeño. Los gráficos llevan tooltips con las medidas de participación (%).

**3. Análisis de cohortes.** Matriz con el mes de cohorte en las filas, el mes de venta en las columnas y la cantidad de ventas o el ingreso como valores, para ver si los clientes vuelven a comprar y comparar cohortes en el tiempo.

## Paso 6: Resumen ejecutivo

### Hallazgos clave

El ingreso total fue de $6,012,502,170 con 8,500 ventas (2023-2024).

Las casas generan más ingreso (37 %, $2.24 mil millones), pero los departamentos dominan en volumen (5,105 ventas, ticket de $362k). Lo comercial tiene el ticket más alto ($1.80M).

Ciudad de México aporta 54 % del ingreso, con un ticket mayor ($786k contra $633k). Bogotá tiene más ventas (4,377 contra 4,123).

El canal Corredor genera 73 % del ingreso y $176M de los $200M en comisiones.

### Métricas principales

- Ingreso total: $6,012,502,170
- Ventas: 8,500
- Ticket promedio: $707,353
- Comisión total: $200,627,166 (3.34 % del ingreso)

### Insights accionables

- El ingreso creció 11.1 % año contra año ($2.85 mil millones a $3.16 mil millones). Vino del volumen (+10.1 % en ventas), porque el ticket casi no cambió.
- Hay estacionalidad, con picos en marzo-abril y septiembre-noviembre.
- 77 % de los clientes (2,421 de 3,139) compró más de una vez, y 61 % de las ventas son recompras.
- Los clientes nuevos cayeron de 2,366 en 2023 a 773 en 2024. El crecimiento depende de la recompra, no de la captación.
- El segmento "Primera vez" concentra 63 % del ingreso, pero 1,454 de sus 1,770 clientes ya compraron varias veces. La etiqueta es estática y subestima la recurrencia.

### Recomendaciones estratégicas

- Reactivar la captación de clientes nuevos, que cayó cerca de 67 %.
- Reforzar la oferta de casas y departamentos según su peso en ingreso y volumen.
- Evaluar la dependencia del canal Corredor, que concentra las comisiones y 73 % del ingreso, e impulsar el canal Directo.
- Armar un programa de retención para las cohortes recientes y compararlas con cohortes antiguas a la misma edad, porque las de 2024 aún tienen poco tiempo para recomprar.
- Concentrar las campañas en los picos de marzo-abril y septiembre-noviembre.

## Entregable

Archivo del dashboard de Power BI (`.pbix`): `https://TU-ENLACE-AL-PBIX`